# All comparisons of tdc tasks

Loading our data:

## AUC Top1 10000 oracle calls table

### Load PMO results from their paper

In [ ]:
def load_yaml_results_fast(filepath):
    """Load results from a YAML file using fast custom parsing.
    
    The YAML file format is:
    SMILES:
    - score
    - oracle_call_number
    
    Returns a list of (score, oracle_call) tuples sorted by oracle_call.
    """
    results = []
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    i = 0
    n = len(lines)
    while i < n:
        line = lines[i]
        if not line.strip():
            i += 1
            continue
        if not line.startswith('-'):
            if i + 2 < n:
                score_line = lines[i + 1].strip()
                oracle_line = lines[i + 2].strip()
                if score_line.startswith('- ') and oracle_line.startswith('- '):
                    try:
                        score = float(score_line[2:])
                        oracle_call = int(oracle_line[2:])
                        results.append((score, oracle_call))
                    except (ValueError, IndexError):
                        pass
                    i += 3
                    continue
        i += 1
    
    results.sort(key=lambda x: x[1])
    return results


# We need to load raw results and recompute AUC with k=1
# The existing cache uses k=10, so we compute fresh from YAML files

def load_pmo_raw_results():
    """Load raw PMO results from YAML files.
    
    Returns dict: model -> task -> list of [(score, oracle_call), ...] for each run
    """
    all_results = defaultdict(lambda: defaultdict(list))
    
    for model in MODEL_MAPPING.keys():
        for task in TASKS:
            pattern = f"results_{model}_{task}_*.yaml"
            for filepath in PMO_RESULTS_DIR.glob(pattern):
                results = load_yaml_results_fast(filepath)
                if results:
                    all_results[model][task].append(results)
    
    return all_results


print("Loading PMO baseline raw results from YAML files...")
pmo_raw_results = load_pmo_raw_results()
n_files = sum(len(runs) for model_data in pmo_raw_results.values() for runs in model_data.values())
print(f"Loaded {n_files} result files for {len(pmo_raw_results)} models")